In [1]:
!nvidia-smi

Sun Nov  9 12:07:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.83                 Driver Version: 572.83         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   44C    P8              3W /  140W |     585MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 计算设备
import torch
from torch import nn

torch.device('cpu'), torch.cuda.device('cuda'), torch.cuda.device('cuda:1')

(device(type='cpu'),
 <torch.cuda.device at 0x1a2be13bb20>)

In [3]:
# 查询可用gpu的数量
torch.cuda.device_count()

1

In [4]:
# 这两个函数允许我们在请求的GPU不存在的情况下运行代码
def try_gpu(i=0):  
    """如果存在，则返回gpu(i)，否则返回cpu()。"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():  
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]。"""
    devices = [
        torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

In [5]:
# 查询张量所在的设备
x = torch.tensor([1,2,3])
x.device

device(type='cpu')

In [7]:
# 存储在GPU上
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [13]:
# 第二个GPU上创建一个随机张量
Y = torch.rand(2, 3, device=try_gpu(0))
Y

tensor([[0.0837, 0.6194, 0.5479],
        [0.4228, 0.9768, 0.9814]], device='cuda:0')

In [14]:
# 要计算X + Y，我们需要决定在哪里执行这个操作
Z = X.cuda(0)
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')


In [15]:
# 现在数据在同一个GPU上（Z和Y都在），我们可以将它们相加
Y + Z

tensor([[1.0837, 1.6194, 1.5479],
        [1.4228, 1.9768, 1.9814]], device='cuda:0')

In [16]:
Z.cuda(0) is Z

True

In [17]:
# 神经网络与GPU
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

net(X)

tensor([[-0.4894],
        [-0.4894]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [18]:
# 确认模型参数存储在同一个GPU上
net[0].weight.data.device

device(type='cuda', index=0)